# Solution 02: your own object
One way to do task 5. The code works for any object: change `PROMPT` and `NAME`, and put your
photos in `my_photos/` and `my_photos_test/`.

In [ ]:
# Setup: installs what is missing (Colab / RunPod) and finds vision.py
import importlib.util, pathlib, subprocess, sys, time, urllib.request
for pkg, mod in [("transformers", "transformers"), ("ultralytics", "ultralytics"),
                 ("segmentation-models-pytorch", "segmentation_models_pytorch")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
if (pathlib.Path.cwd().parent / "vision.py").exists():   # opened from vision/notebooks or vision/sol
    sys.path.insert(0, str(pathlib.Path.cwd().parent))
elif not pathlib.Path("vision.py").exists():            # Colab: fetch it from the course repo
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/zhaw-physical-ai/cas-pair/main/vision/vision.py", "vision.py")
from vision import *
device = get_device()
print("device:", device)

## Your photos
Put 30-50 photos of your object in `my_photos/`, and 10 photos taken in a *different place* in
`my_photos_test/` (Colab: upload a zip in the file panel, then `!unzip -q photos.zip -d my_photos`).

In [ ]:
PROMPT = "a screwdriver."     # your object: lower case, one noun phrase, ends with a period
NAME = "screwdriver"

photos = load_folder("my_photos")
test_photos = load_folder("my_photos_test")
print(len(photos), "training photos,", len(test_photos), "test photos")
fig, axs = plt.subplots(1, 6, figsize=(24, 4))
for ax, im in zip(axs, photos):
    show(ax, im)

In [ ]:
boxer = TextBoxer()     # Grounding DINO tiny, ~700 MB on first use
sam = SamLabeler()
boxes, scores = boxer(photos[0], PROMPT)
fig, ax = plt.subplots(figsize=(6, 6))
show(ax, photos[0], sam(photos[0], boxes=boxes)[0] if boxes else None, boxes=boxes, title=f"scores {np.round(scores, 2)}")

## Label
For every photo: boxes from the prompt, a mask from SAM.

In [ ]:
labels = []
for im in photos:
    boxes, scores = boxer(im, PROMPT)
    if len(boxes) == 0:
        labels.append(np.zeros((0, *im.shape[:2]), bool))
        continue
    best = [boxes[int(np.argmax(scores))]]          # one object per photo: keep the most confident box
    labels.append(sam(im, boxes=best)[0])
print(sum(len(l) == 0 for l in labels), "of", len(photos), "photos without a box")

## Check every label
This is the human part of the pipeline: look at every label and drop the bad ones.

In [ ]:
cols = 6
rows = int(np.ceil(len(photos) / cols))
fig, axs = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), squeeze=False)
for k, ax in enumerate(axs.flat):
    if k < len(photos):
        show(ax, photos[k], labels[k] if len(labels[k]) else None, title=str(k))
    else:
        ax.axis("off")
plt.tight_layout()

In [ ]:
DROP = []      # indices of bad labels, from the grid above
keep = [k for k in range(len(photos)) if k not in DROP and len(labels[k])]
print(f"training on {len(keep)} of {len(photos)} photos")

**Result:** Typical failures: the prompt finds a similar object (a pen for a screwdriver) or the whole scene, the object is cut by the image edge, or SAM's mask leaks into a background of the same colour. Each dropped photo is one the model will not learn from, so a better prompt or threshold that saves photos is worth trying.

## Train and test
Train YOLO on the kept photos (see task 3), then try it on `test_photos`: taken in a place the
model has never seen.

In [ ]:
yaml = write_yolo_dataset(f"yolo_{NAME}", [photos[k] for k in keep], [labels[k] for k in keep], name=NAME)
model = train_yolo(yaml, name=NAME, epochs=50)

In [ ]:
fig, axs = plt.subplots(2, 5, figsize=(25, 10), squeeze=False)
for ax, im in zip(axs.flat, test_photos):
    masks, boxes, conf = yolo_predict(model, im)
    show(ax, im, masks, boxes=boxes, title=f"conf {np.round(conf, 2)}")
plt.tight_layout()

# a number, with a caveat: the reference labels come from the same text prompt + SAM
ref = []
for im in test_photos:
    b, s = boxer(im, PROMPT)
    ref.append(sam(im, boxes=[b[int(np.argmax(s))]])[0].any(0) if len(b) else np.zeros(im.shape[:2], bool))
print(f"test IoU against the automatic labels: {np.mean([iou(yolo_predict(model, im)[0].any(0), r) for im, r in zip(test_photos, ref)]):.3f}")

**Result:** With 30-50 photos the model usually finds the object on a plain new background and fails on busy ones, in different light, or at unusual angles: it has partly learned the backgrounds of the training photos. The test number is optimistic in a hidden way: the reference labels come from the same prompt and SAM, so their mistakes do not count as errors. To trust it, check the test labels by eye, or draw them yourself.

## Advanced
- How many photos did you really need? Train on half of `keep` and compare on the test photos.
- The test labels also came from SAM. How sure are you of your test numbers?